In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from google.colab import files
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize
import math
from itertools import combinations

uploaded = files.upload()  # Upload your file in Colab

file_name = list(uploaded.keys())[0]  # Get the name of the uploaded file
data = pd.read_excel(file_name)


X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values


scaler = StandardScaler()
X = scaler.fit_transform(X)


sss = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
for train_index, test_index in sss.split(X, y):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]


smote = SMOTE(k_neighbors=1)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)


# Initialize classifiers
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC(probability=True, random_state=42)
}

# Train, predict, and evaluate each model
for name, model in models.items():
    print(f"\n=== {name} ===")
    # Train the model
    model.fit(X_resampled, y_resampled)

    # Make predictions
    y_pred = model.predict(X_test)
    # Step 1: Predict probabilities for the test set
    y_pred_proba = model.predict_proba(X_test)

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')


    # Step 1: Predict probabilities for the test set
    n_classes = 6
    y_test_binarized = label_binarize(y_test, classes=range(len(set(y))))
    y_pred_binarized = label_binarize(y_pred, classes=range(len(set(y))))
    # Step 2: Binarize the labels for multi-class AUC computation
    y_test_binarized = label_binarize(y_test, classes=list(range(6)))
    # Step 3: Calculate pairwise AUCs
    pairwise_aucs = []
    class_pairs = list(combinations(range(6), 2))  # Generate all class pairs (C(6, 2) = 15)
    pairwise_auc_matrix = np.zeros((n_classes, n_classes))


    for class_a, class_b in class_pairs:
      # Filter probabilities and true labels for the two classes
      mask = (y_test == class_a) | (y_test == class_b)
      y_true_pair = y_test[mask]
      # Use boolean indexing for rows and integer indexing for columns separately
      y_proba_pair = y_pred_proba[mask][:, [class_a, class_b]]  # Keep only the columns for class_a and class_b

      # Re-label for binary classification
      y_true_binary = (y_true_pair == class_b).astype(int)  # 1 for class_b, 0 for class_a

      # Calculate AUC for the pair
      auc = roc_auc_score(y_true_binary, y_proba_pair[:, 1])  # Use probabilities for class_b
      pairwise_aucs.append(auc)
      pairwise_auc_matrix[class_a, class_b] = auc
      pairwise_auc_matrix[class_b, class_a] = auc


    # Step 4: Compute the multi-class AUC using the equiangular area approach
    q = len(pairwise_aucs)  # Number of pairwise AUCs (should be 15)
    r = np.mean(pairwise_aucs)  # Mean pairwise AUC, used for normalization

    # Maximum area calculation (equiangular polygon)
    equiangular_area = (q / 2) * (r ** 2) * math.sin((2 * math.pi) / q)
    scaled_auc = (equiangular_area-0.7626)/(3.0505-0.7626)
    # Display results
    print("Model name:")
    print(name)
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print("Multi-Class AUC (equiangular area):", scaled_auc)



Saving 594_eating.xlsx to 594_eating (1).xlsx

=== Random Forest ===
Model name:
Random Forest
Accuracy: 0.6111
Precision: 0.5910
Recall: 0.6111
F1 Score: 0.5986
Multi-Class AUC (equiangular area): 0.8193677887212648

=== SVM ===
Model name:
SVM
Accuracy: 0.4167
Precision: 0.4414
Recall: 0.4167
F1 Score: 0.4024
Multi-Class AUC (equiangular area): 0.5331874477060937


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
